# 03.5 — CRF for Sequence Labeling

**Problem it solves:** POS tagging, NER, chunking — labeling every token in a sequence. The challenge: labels have dependencies (B-PER must be followed by I-PER, not I-ORG).

**Why CRF beats naive per-token classification:** A CRF models the joint probability P(y1...yn | x1...xn) — it can enforce label consistency constraints globally. An LSTM-CRF is still state-of-the-art for NER without transformers.

---

In [ ]:
import numpy as np
from collections import defaultdict

# ---- Viterbi Algorithm ----
# The core decoding algorithm for sequence models.
# Find the most likely tag sequence given observations.

def viterbi(obs: list, states: list, 
            start_p: dict, trans_p: dict, emit_p: dict) -> tuple:
    """
    Viterbi decoding for HMM/CRF-like models.
    
    obs: list of observed tokens
    states: list of possible tags
    start_p: P(tag at position 0)
    trans_p: P(tag[t] | tag[t-1])
    emit_p: P(word | tag)
    
    Returns: (best_score, best_path)
    """
    n = len(obs)
    k = len(states)
    state_idx = {s: i for i, s in enumerate(states)}
    
    # DP tables (log space to avoid underflow)
    viterbi_dp = np.full((n, k), -np.inf)
    backpointer = np.zeros((n, k), dtype=int)
    
    eps = 1e-10
    
    # Initialization: t=0
    for s, tag in enumerate(states):
        sp = start_p.get(tag, eps)
        ep = emit_p.get(tag, {}).get(obs[0], eps)
        viterbi_dp[0][s] = np.log(sp) + np.log(ep)
    
    # Recursion
    for t in range(1, n):
        for s, tag in enumerate(states):
            ep = np.log(emit_p.get(tag, {}).get(obs[t], eps))
            # Best previous state
            scores = [
                viterbi_dp[t-1][s2] + np.log(trans_p.get(prev_tag, {}).get(tag, eps)) + ep
                for s2, prev_tag in enumerate(states)
            ]
            viterbi_dp[t][s] = max(scores)
            backpointer[t][s] = np.argmax(scores)
    
    # Backtrace
    best_last = int(np.argmax(viterbi_dp[n-1]))
    best_path = [best_last]
    for t in range(n-1, 0, -1):
        best_path.insert(0, backpointer[t][best_path[0]])
    
    return viterbi_dp[n-1][best_last], [states[i] for i in best_path]

# POS tagging with HMM
# (Simplified: in practice learn these from a tagged corpus)
states = ['NN', 'VB', 'DT', 'JJ', 'IN']  # noun, verb, determiner, adj, preposition

start_probs = {'DT': 0.4, 'NN': 0.2, 'JJ': 0.1, 'VB': 0.2, 'IN': 0.1}

trans_probs = {
    'DT': {'NN': 0.5, 'JJ': 0.4, 'VB': 0.05, 'DT': 0.02, 'IN': 0.03},
    'JJ': {'NN': 0.7, 'JJ': 0.2, 'VB': 0.05, 'DT': 0.03, 'IN': 0.02},
    'NN': {'VB': 0.4, 'NN': 0.1, 'IN': 0.3, 'DT': 0.1, 'JJ': 0.1},
    'VB': {'DT': 0.4, 'NN': 0.3, 'IN': 0.2, 'JJ': 0.05, 'VB': 0.05},
    'IN': {'DT': 0.5, 'NN': 0.3, 'JJ': 0.15, 'VB': 0.03, 'IN': 0.02},
}

emit_probs = {
    'DT': {'the': 0.5, 'a': 0.3, 'an': 0.2},
    'NN': {'dog': 0.15, 'cat': 0.15, 'fox': 0.15, 'house': 0.1, 'man': 0.1, 'city': 0.1, 'time': 0.1, 'tree': 0.1, 'hill': 0.05},
    'VB': {'runs': 0.2, 'jumps': 0.2, 'chases': 0.2, 'sees': 0.2, 'ate': 0.1, 'slept': 0.1},
    'JJ': {'quick': 0.25, 'brown': 0.25, 'lazy': 0.25, 'big': 0.15, 'old': 0.1},
    'IN': {'over': 0.25, 'under': 0.25, 'near': 0.25, 'on': 0.25},
}

sentences = [
    ['the', 'quick', 'brown', 'fox', 'jumps', 'over', 'the', 'lazy', 'dog'],
    ['a', 'big', 'dog', 'chases', 'the', 'cat'],
]

for sent in sentences:
    # Filter to known words only
    known = [w if any(w in emit_probs[t] for t in states) else 'dog' 
             for w in sent]
    score, tags = viterbi(known, states, start_probs, trans_probs, emit_probs)
    print(f"Sentence: {sent}")
    print(f"Tags:     {tags}")
    print(f"Score:    {score:.2f}")
    print()

In [ ]:
# ---- BIO tagging for NER ----
# B-TAG = Beginning of entity of type TAG
# I-TAG = Inside entity of type TAG (continues previous B or I)
# O    = Outside (not part of any entity)

# IOB constraints that a CRF can enforce:
# - I-PER must follow B-PER or I-PER (not B-ORG or O)
# - B-* can follow anything
# - O can follow anything

# Without CRF (naive per-token classifier), you might get:
# [B-PER, I-ORG, I-PER] which is invalid
# With CRF transition scores, invalid transitions get -inf score

bio_tags = ['O', 'B-PER', 'I-PER', 'B-ORG', 'I-ORG', 'B-LOC', 'I-LOC']

# CRF transition constraints
def is_valid_bio_transition(prev_tag: str, curr_tag: str) -> bool:
    """Check if a BIO tag transition is valid."""
    if curr_tag.startswith('I-'):
        entity_type = curr_tag[2:]
        # I-X must follow B-X or I-X
        if prev_tag != f'B-{entity_type}' and prev_tag != f'I-{entity_type}':
            return False
    return True

print("BIO transition validity:")
test_transitions = [
    ('O', 'B-PER'), ('B-PER', 'I-PER'), ('I-PER', 'I-PER'),
    ('B-PER', 'I-ORG'),  # INVALID
    ('O', 'I-PER'),       # INVALID
    ('B-ORG', 'I-ORG'), ('I-ORG', 'O'), ('O', 'O'),
]
for prev, curr in test_transitions:
    valid = is_valid_bio_transition(prev, curr)
    marker = '✓' if valid else '✗ INVALID'
    print(f"  {prev:8} -> {curr:8}  {marker}")

In [ ]:
# Feature engineering for a linear CRF
# CRFs use hand-crafted features: word shape, prefix/suffix, capitalization etc.
# Neural CRFs (BiLSTM-CRF, BERT-CRF) learn features automatically.

def extract_features(tokens: list[str], i: int) -> dict:
    """Extract features for token at position i."""
    word = tokens[i]
    features = {
        'word': word.lower(),
        'word.isupper': word.isupper(),
        'word.istitle': word.istitle(),          # Capitalized
        'word.isdigit': word.isdigit(),
        'word.prefix2': word[:2].lower(),
        'word.prefix3': word[:3].lower(),
        'word.suffix2': word[-2:].lower(),
        'word.suffix3': word[-3:].lower(),
        'word.has_hyphen': '-' in word,
        'word.has_digit': any(c.isdigit() for c in word),
    }
    
    # Context features
    if i > 0:
        prev = tokens[i-1]
        features['prev_word'] = prev.lower()
        features['prev_istitle'] = prev.istitle()
    else:
        features['BOS'] = True  # beginning of sentence
    
    if i < len(tokens) - 1:
        next_w = tokens[i+1]
        features['next_word'] = next_w.lower()
        features['next_istitle'] = next_w.istitle()
    else:
        features['EOS'] = True
    
    return features

# Test on NER example
sentence = ['Barack', 'Obama', 'was', 'born', 'in', 'Honolulu', 'Hawaii']
gold_tags = ['B-PER', 'I-PER', 'O', 'O', 'O', 'B-LOC', 'I-LOC']

print("Features extracted for NER:")
for i, (token, tag) in enumerate(zip(sentence, gold_tags)):
    feats = extract_features(sentence, i)
    informative = {k:v for k,v in feats.items() 
                   if v not in (False, '') and k not in ['word']}
    print(f"  {token:12} ({tag:8})  features: {informative}")

## Summary

**Viterbi + dynamic programming** is the key algorithmic idea — used in HMMs, CRFs, and as the final decoding layer in many neural NER systems.

**CRF advantage over naive per-token classification:**
- Enforces label consistency (I-PER can't follow B-ORG)
- Considers the whole sequence when making decisions

**What replaced CRFs:**
- BiLSTM-CRF: uses LSTM to build contextual representations, CRF for decoding
- BERT-CRF: even better representations, CRF still used for label consistency
- In practice: BERT fine-tuned for token classification (without CRF) works nearly as well

---
**Module 03 complete. Next: Module 04 — Word Vectors**